# Homework 2

For this assignment I will experiment with Yolov8n and Faster R-CNN detection models and compare their performance. A sample traffic video will be used to test the models but a coco data set will be used to evaluate the models. Also Onnx runtime and OpenVino core will also be used to improve the performance of the models.

Video by Mike Bird from Pexels: https://www.pexels.com/video/traffic-flow-in-the-highway-2103099/

## Project Setup

In [2]:
import os
import shutil
import time
import json
from datetime import datetime
from pathlib import Path
from typing import List, Tuple

import cv2
import matplotlib.colors as mcolors
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import openvino as ov
import torch
import torchvision
from PIL import Image, ImageDraw, ImageFont
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from tqdm import tqdm
from torchvision import transforms
from torchvision.io import read_image
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights, fasterrcnn_resnet50_fpn

from ultralytics import YOLO
from ultralytics.utils import ASSETS, yaml_load
from ultralytics.utils.checks import check_requirements, check_yaml

from extractframefromvideo import extract_key_frames


In [ ]:
# extract frames from video
extract_key_frames(
    "video.mp4", 
    'output/extracted', 
    target_size=(800, 800),
    extraction_method="both"
)

Exporting Yolov8n as Onnx and OpenVino formats

In [ ]:
# Make sure 'models' directory exists
os.makedirs('models', exist_ok=True)

# Load the YOLOv8n model
model = YOLO('yolov8n.pt')

# Export to OpenVINO
model.export(format='openvino', imgsz=640, device='cpu', half=False)
openvino_src = 'yolov8n_openvino_model'
openvino_dst = 'models/yolov8n_openvino_model'
shutil.move(openvino_src, openvino_dst)

# Export to ONNX
model.export(format='onnx', imgsz=640, device='cpu', half=False, opset=12)
onnx_src = 'yolov8n.onnx'
onnx_dst = 'models/yolov8n.onnx'
shutil.move(onnx_src, onnx_dst)

# Save the original PyTorch model (.pt)
shutil.copy('yolov8n.pt', 'models/yolov8n.pt')

print("Exported models saved to 'models/' folder!")

Exporting Faster R-CNN as Onnx and OpenVino formats

In [59]:
# Export the Faster R-CNN model to ONNX format

# Load the model
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
model.eval()

# Create a wrapper class to handle the list input
class FasterRCNNWrapper(torch.nn.Module):
    def __init__(self, model):
        super(FasterRCNNWrapper, self).__init__()
        self.model = model
        
    def forward(self, x):
        # In ONNX export, we can only handle a single image at a time
        # We'll just use the model in evaluation mode, which returns a list of dictionaries
        return self.model([x])[0]

# Create the wrapper model
wrapped_model = FasterRCNNWrapper(model)

# Create a single tensor input (ONNX export can't handle lists of tensors)
# dummy_input = torch.rand(3, 300, 400)
dummy_input = torch.rand(3, 640, 480)

# Export to ONNX
torch.onnx.export(
    wrapped_model,
    dummy_input,
    "models/faster_rcnn.onnx",
    opset_version=12,
    input_names=["input"],
    output_names=["boxes", "labels", "scores"],
    dynamic_axes={
        "input": {1: "height", 2: "width"},
        "boxes": {0: "num_boxes", 1: "coordinates"},
        "labels": {0: "num_boxes"},
        "scores": {0: "num_boxes"}
    }
)


In [60]:
ov_model = ov.convert_model("models/faster_rcnn.onnx")
ov.save_model(ov_model, "models/fasterrcnn_openvino_model/faster_rcnn_model.xml")

## Inference Yolov8n model

In [11]:
# https://docs.ultralytics.com/models/yolov8/#yolov8-usage-examples

output = "output/results/yolo/yolov8n"  # Output Folder
images = "output/extracted"  # Path to image folder
os.makedirs(output, exist_ok=True) # Create output directory if it doesn't exist

model = YOLO("yolov8n.pt")  # Load a pretrained YOLOv8 model

# Start timer
start_time = time.time()

# for each image
for image in os.listdir(images):
    image_path = os.path.join(images, image)
    
    # check if the file is an image
    if not image.lower().endswith(('.png', '.jpg', '.jpeg')):
        # print(f"Skipping non-image file: {image}")
        continue
    
    result = model(image_path)
    
    # get the input image name
    input_image_name = image.split("/")[-1].split(".")[0]
    output_image_name = f"{output}/{input_image_name}.jpg"
    
    # Save result to output directory
    result[0].save(output_image_name)
    
# Print the time taken for inference
elapsed_time_ms = (time.time() - start_time) * 1000
print(f"Time taken for inference: {elapsed_time_ms:.2f} ms")

# Counter for JPEG images
jpeg_images = len([f for f in os.listdir(images) if f.lower().endswith('.jpg')])

# Print number of images
print(f"Number of images: {jpeg_images}")

# Print average time per image
average_time_per_image_ms = elapsed_time_ms / jpeg_images
print(f"Average time per image: {average_time_per_image_ms:.2f} ms")


image 1/1 /Users/phuocle/Desktop/Class Archive/Classes/CMPE 258/Homework/Homework 2/output/extracted/frame_0-00-27.000_interval.jpg: 384x640 20 cars, 4 trucks, 45.0ms
Speed: 1.0ms preprocess, 45.0ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /Users/phuocle/Desktop/Class Archive/Classes/CMPE 258/Homework/Homework 2/output/extracted/frame_0-00-19.000_interval.jpg: 384x640 19 cars, 3 trucks, 42.1ms
Speed: 1.4ms preprocess, 42.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /Users/phuocle/Desktop/Class Archive/Classes/CMPE 258/Homework/Homework 2/output/extracted/frame_0-00-42.000_interval.jpg: 384x640 23 cars, 1 truck, 45.6ms
Speed: 1.2ms preprocess, 45.6ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /Users/phuocle/Desktop/Class Archive/Classes/CMPE 258/Homework/Homework 2/output/extracted/frame_0-00-38.000_interval.jpg: 384x640 26 cars, 1 truck, 41.2ms
Speed: 0.9ms preprocess, 41.2ms inferen

In [ ]:
model = YOLO("models/yolov8n.pt", task="detect")
results = model.val(data="coco128.yaml", task="detect", conf=0.25, iou=0.70, device="cpu", save_json=True, plots=True)

Ultralytics 8.3.118 🚀 Python-3.13.3 torch-2.7.0 CPU (Apple M2)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1487.6±699.2 MB/s, size: 47.6 KB)


val: Scanning /Users/phuocle/Desktop/untitled folder/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]
/Users/phuocle/Desktop/Class Archive/Classes/CMPE 258/Homework/Homework 2/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:15<00:00,  1.90s/it]


                   all        128        929      0.677      0.499       0.62      0.493
                person         61        254      0.813      0.669      0.778      0.611
               bicycle          3          6      0.667      0.333      0.499       0.43
                   car         12         46      0.909      0.217      0.571      0.398
            motorcycle          4          5      0.667        0.8      0.825      0.704
              airplane          5          6        0.8      0.667        0.8      0.612
                   bus          5          7        0.5      0.714      0.786      0.759
                 train          3          3      0.667      0.667      0.777      0.677
                 truck          5         12          1       0.25      0.625      0.444
                  boat          2          6       0.25      0.167       0.27       0.15
         traffic light          4         14      0.667      0.143      0.429      0.386
             stop sig

## Inference Yolov8n using Onnx runtime

This code was taken and adapted from https://github.com/ultralytics/ultralytics/blob/main/examples/YOLOv8-ONNXRuntime/main.py

In [12]:
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

class OnnxModel:
    """
    YOLOv8 object detection model class for handling inference and visualization.

    This class provides functionality to load a YOLOv8 ONNX model, perform inference on images,
    and visualize the detection results.

    Attributes:
        onnx_model (str): Path to the ONNX model file.
        input_image (str): Path to the input image file.
        confidence_thres (float): Confidence threshold for filtering detections.
        iou_thres (float): IoU threshold for non-maximum suppression.
        classes (List[str]): List of class names from the COCO dataset.
        color_palette (np.ndarray): Random color palette for visualizing different classes.
        input_width (int): Width dimension of the model input.
        input_height (int): Height dimension of the model input.
        img (np.ndarray): The loaded input image.
        img_height (int): Height of the input image.
        img_width (int): Width of the input image.
    """

    def __init__(self, onnx_model: str, confidence_thres: float, iou_thres: float, input_image: str = ""):
        """
        Initialize an instance of the YOLOv8 class.

        Args:
            onnx_model (str): Path to the ONNX model.
            input_image (str): Path to the input image.
            confidence_thres (float): Confidence threshold for filtering detections.
            iou_thres (float): IoU threshold for non-maximum suppression.
        """
        self.onnx_model = onnx_model
        self.input_image = input_image
        self.confidence_thres = confidence_thres
        self.iou_thres = iou_thres

        # Load the class names from the COCO dataset
        self.classes = yaml_load(check_yaml("coco8.yaml"))["names"]

        # Generate a color palette for the classes
        np.random.seed(42)
        self.color_palette = np.random.uniform(0, 255, size=(len(self.classes), 3))
        
    def set_image(self, input_image: str) -> None:
        """
        Set the input image for inference.

        Args:
            input_image (str): Path to the input image.
        """
        self.input_image = input_image

    def letterbox(self, img: np.ndarray, new_shape: Tuple[int, int] = (640, 640)) -> Tuple[np.ndarray, Tuple[int, int]]:
        """
        Resize and reshape images while maintaining aspect ratio by adding padding.

        Args:
            img (np.ndarray): Input image to be resized.
            new_shape (Tuple[int, int]): Target shape (height, width) for the image.

        Returns:
            (np.ndarray): Resized and padded image.
            (Tuple[int, int]): Padding values (top, left) applied to the image.
        """
        shape = img.shape[:2]  # current shape [height, width]

        # Scale ratio (new / old)
        r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])

        # Compute padding
        new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
        dw, dh = (new_shape[1] - new_unpad[0]) / 2, (new_shape[0] - new_unpad[1]) / 2  # wh padding

        if shape[::-1] != new_unpad:  # resize
            img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
        top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
        left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
        img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114))

        return img, (top, left)

    def draw_detections(self, img: np.ndarray, box: List[float], score: float, class_id: int) -> None:
        """
        Draw bounding boxes and labels on the input image based on the detected objects.

        Args:
            img (np.ndarray): The input image to draw detections on.
            box (List[float]): Detected bounding box coordinates [x, y, width, height].
            score (float): Confidence score of the detection.
            class_id (int): Class ID for the detected object.
        """
        # Extract the coordinates of the bounding box
        x1, y1, w, h = box

        # Retrieve the color for the class ID
        color = self.color_palette[class_id]

        # Draw the bounding box on the image
        cv2.rectangle(img, (int(x1), int(y1)), (int(x1 + w), int(y1 + h)), color, 2)

        # Create the label text with class name and score
        label = f"{self.classes[class_id]}: {score:.2f}"

        # Calculate the dimensions of the label text
        (label_width, label_height), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)

        # Calculate the position of the label text
        label_x = x1
        label_y = y1 - 10 if y1 - 10 > label_height else y1 + 10

        # Draw a filled rectangle as the background for the label text
        cv2.rectangle(
            img, (label_x, label_y - label_height), (label_x + label_width, label_y + label_height), color, cv2.FILLED
        )

        # Draw the label text on the image
        cv2.putText(img, label, (label_x, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)

    def preprocess(self) -> Tuple[np.ndarray, Tuple[int, int]]:
        """
        Preprocess the input image before performing inference.

        This method reads the input image, converts its color space, applies letterboxing to maintain aspect ratio,
        normalizes pixel values, and prepares the image data for model input.

        Returns:
            (np.ndarray): Preprocessed image data ready for inference with shape (1, 3, height, width).
            (Tuple[int, int]): Padding values (top, left) applied during letterboxing.
        """
        # Read the input image using OpenCV
        self.img = cv2.imread(self.input_image)

        # Get the height and width of the input image
        self.img_height, self.img_width = self.img.shape[:2]

        # Convert the image color space from BGR to RGB
        img = cv2.cvtColor(self.img, cv2.COLOR_BGR2RGB)

        img, pad = self.letterbox(img, (self.input_width, self.input_height))

        # Normalize the image data by dividing it by 255.0
        image_data = np.array(img) / 255.0

        # Transpose the image to have the channel dimension as the first dimension
        image_data = np.transpose(image_data, (2, 0, 1))  # Channel first

        # Expand the dimensions of the image data to match the expected input shape
        image_data = np.expand_dims(image_data, axis=0).astype(np.float32)

        # Return the preprocessed image data
        return image_data, pad

    def postprocess(self, input_image: np.ndarray, output: List[np.ndarray], pad: Tuple[int, int]) -> np.ndarray:
        """
        Perform post-processing on the model's output to extract and visualize detections.

        This method processes the raw model output to extract bounding boxes, scores, and class IDs.
        It applies non-maximum suppression to filter overlapping detections and draws the results on the input image.

        Args:
            input_image (np.ndarray): The input image.
            output (List[np.ndarray]): The output arrays from the model.
            pad (Tuple[int, int]): Padding values (top, left) used during letterboxing.

        Returns:
            (np.ndarray): The input image with detections drawn on it.
        """
        # Transpose and squeeze the output to match the expected shape
        outputs = np.transpose(np.squeeze(output[0]))

        # Get the number of rows in the outputs array
        rows = outputs.shape[0]

        # Lists to store the bounding boxes, scores, and class IDs of the detections
        boxes = []
        scores = []
        class_ids = []

        # Calculate the scaling factors for the bounding box coordinates
        gain = min(self.input_height / self.img_height, self.input_width / self.img_width)
        outputs[:, 0] -= pad[1]
        outputs[:, 1] -= pad[0]

        # Iterate over each row in the outputs array
        for i in range(rows):
            # Extract the class scores from the current row
            classes_scores = outputs[i][4:]

            # Find the maximum score among the class scores
            max_score = np.amax(classes_scores)

            # If the maximum score is above the confidence threshold
            if max_score >= self.confidence_thres:
                # Get the class ID with the highest score
                class_id = np.argmax(classes_scores)

                # Extract the bounding box coordinates from the current row
                x, y, w, h = outputs[i][0], outputs[i][1], outputs[i][2], outputs[i][3]

                # Calculate the scaled coordinates of the bounding box
                left = int((x - w / 2) / gain)
                top = int((y - h / 2) / gain)
                width = int(w / gain)
                height = int(h / gain)

                # Add the class ID, score, and box coordinates to the respective lists
                class_ids.append(class_id)
                scores.append(max_score)
                boxes.append([left, top, width, height])

        # Apply non-maximum suppression to filter out overlapping bounding boxes
        indices = cv2.dnn.NMSBoxes(boxes, scores, self.confidence_thres, self.iou_thres)

        # Iterate over the selected indices after non-maximum suppression
        for i in indices:
            # Get the box, score, and class ID corresponding to the index
            box = boxes[i]
            score = scores[i]
            class_id = class_ids[i]

            # Draw the detection on the input image
            self.draw_detections(input_image, box, score, class_id)

        # Return the modified input image
        return input_image

    def main(self) -> np.ndarray:
        """
        Perform inference using an ONNX model and return the output image with drawn detections.

        Returns:
            (np.ndarray): The output image with drawn detections.
        """
        # Create an inference session using the ONNX model and specify execution providers
        session = ort.InferenceSession(self.onnx_model, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])

        # Get the model inputs
        model_inputs = session.get_inputs()

        # Store the shape of the input for later use
        input_shape = model_inputs[0].shape
        self.input_width = input_shape[2]
        self.input_height = input_shape[3]

        # Preprocess the image data
        img_data, pad = self.preprocess()

        # Run inference using the preprocessed image data
        outputs = session.run(None, {model_inputs[0].name: img_data})

        # Perform post-processing on the outputs to obtain output image.
        return self.postprocess(self.img, outputs, pad)  # output image


In [13]:
model = "models/yolov8n.onnx"  # ONNX model
images = "output/extracted"  # Path to image folder
output = "output/results/yolo/onnx"  # Output Folder
conf_thres = 0.25  # Confidence threshold
iou_thres = 0.7  # NMS IoU threshold


# Check the requirements and select the appropriate backend (CPU or GPU)
check_requirements("onnxruntime-gpu" if torch.cuda.is_available() else "onnxruntime")

# create output directory if it doesn't exist
os.makedirs(output, exist_ok=True)

# Create an instance of the YOLOv8 class with the specified arguments
detection = OnnxModel(model, conf_thres, iou_thres)

# Start timer
start_time = time.time()

# for each image
for image in os.listdir(images):
    # get the full path of the image
    img = os.path.join(images, image)
    
    # check if the file is an image
    if not img.lower().endswith(('.png', '.jpg', '.jpeg')):
        # print(f"Skipping non-image file: {img}")
        continue

    # Set the input image for the detection instance
    detection.set_image(img)

    # Perform object detection and obtain the output image
    output_image = detection.main()
    
    # get the input image name
    input_image_name = img.split("/")[-1].split(".")[0]
    output_image_name = f"{output}/{input_image_name}.jpg"

    # save image
    cv2.imwrite(output_image_name, output_image)

# Print the time taken for inference
elapsed_time_ms = (time.time() - start_time) * 1000
print(f"Time taken for inference: {elapsed_time_ms:.2f} ms")

# Counter for JPEG images
jpeg_images = len([f for f in os.listdir(images) if f.lower().endswith('.jpg')])

# Print number of images
print(f"Number of images: {jpeg_images}")

# Print average time per image
average_time_per_image_ms = elapsed_time_ms / jpeg_images
print(f"Average time per image: {average_time_per_image_ms:.2f} ms")

Time taken for inference: 4335.88 ms
Number of images: 60
Average time per image: 72.26 ms


In [10]:
model = YOLO("models/yolov8n.onnx", task="detect")
results = model.val(data="coco128.yaml", task="detect", conf=0.25, iou=0.70, device="cpu", save_json=True, plots=True)

Ultralytics 8.3.118 🚀 Python-3.13.3 torch-2.7.0 CPU (Apple M2)
Loading models/yolov8n.onnx for ONNX Runtime inference...
Using ONNX Runtime CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 181.1±80.4 MB/s, size: 36.8 KB)


val: Scanning /Users/phuocle/Desktop/untitled folder/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]
/Users/phuocle/Desktop/Class Archive/Classes/CMPE 258/Homework/Homework 2/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 128/128 [00:05<00:00, 23.76it/s]


                   all        128        929      0.656      0.511      0.619        0.5
                person         61        254      0.805      0.665      0.777      0.617
               bicycle          3          6        0.5      0.167      0.374      0.374
                   car         12         46        0.9      0.196      0.558      0.448
            motorcycle          4          5      0.667        0.8      0.825      0.721
              airplane          5          6        0.8      0.667        0.8      0.626
                   bus          5          7      0.556      0.714      0.771        0.7
                 train          3          3      0.667      0.667      0.777      0.727
                 truck          5         12       0.75       0.25       0.53      0.366
                  boat          2          6        0.2      0.167      0.249      0.133
         traffic light          4         14      0.667      0.143      0.429      0.405
             stop sig

## Inference Yolov8n using OpenVino Core

Some of the code was taken and adapted from https://github.com/openvinotoolkit/openvino_notebooks/tree/latest/notebooks/yolov8-optimization

In [19]:
class OpenVINOModel:
    def __init__(self, models_dir: Path, det_model_name: str, device: str = "CPU", output_dir: str = "output"):
        self.models_dir = models_dir
        self.det_model_name = det_model_name
        self.device = device
        self.output_dir = Path(output_dir)
        self.core = ov.Core()

        # Ensure the models directory exists
        os.makedirs(self.models_dir, exist_ok=True)
        
        # Load the original YOLO model
        self.det_model = YOLO(models_dir / f"{det_model_name}.pt")
        _ = self.det_model.predict(source=np.zeros((640, 640, 3), dtype=np.uint8), verbose=False)

        # Prepare OpenVINO model path
        self.det_model_path = models_dir / f"{det_model_name}_openvino_model/{det_model_name}.xml"

        # Export if OpenVINO model doesn't exist
        if not self.det_model_path.exists():
            self.det_model.export(format="openvino", dynamic=True, half=True)

        # Load and compile the OpenVINO model
        self._load_openvino_model()

        # Override YOLO's predictor to use OpenVINO inference
        self.det_model.predictor.inference = self._infer
        self.det_model.predictor.model.pt = False

    def _load_openvino_model(self):
        self.ov_model = self.core.read_model(self.det_model_path)

        ov_config = {}
        if self.device != "CPU":
            # Force input shape for non-CPU devices
            self.ov_model.reshape({0: [1, 3, 640, 640]})

        if "GPU" in self.device or ("AUTO" in self.device and "GPU" in self.core.available_devices):
            ov_config = {"GPU_DISABLE_WINOGRAD_CONVOLUTION": "YES"}

        self.det_compiled_model = self.core.compile_model(self.ov_model, self.device, ov_config)

    def _infer(self, *args):
        result = self.det_compiled_model(args)
        return torch.from_numpy(result[0])

    def detect(self, image_path: str):
        image_name = image_path.split("/")[-1]
        res = self.det_model(image_path)
        img = Image.fromarray(res[0].plot()[:, :, ::-1])
        img.save(self.output_dir / image_name)

In [20]:
modelPath = "models"  # Model Folder
output = "output/results/yolo/openvino"  # Output Folder
images = "output/extracted"  # Path to image folder
os.makedirs(output, exist_ok=True) # Create output directory if it doesn't exist

model = OpenVINOModel(models_dir=Path(modelPath), det_model_name="yolov8n", device="CPU", output_dir=output)

# Start timer
start_time = time.time()

# for each image
for image in os.listdir(images):
    image_path = os.path.join(images, image)
    
    # check if the file is an image
    if not image.lower().endswith(('.png', '.jpg', '.jpeg')):
        # print(f"Skipping non-image file: {image}")
        continue
    
    model.detect(image_path)

# Print the time taken for inference
elapsed_time_ms = (time.time() - start_time) * 1000
print(f"Time taken for inference: {elapsed_time_ms:.2f} ms")

# Counter for JPEG images
jpeg_images = len([f for f in os.listdir(images) if f.lower().endswith('.jpg')])

# Print number of images
print(f"Number of images: {jpeg_images}")

# Print average time per image
average_time_per_image_ms = elapsed_time_ms / jpeg_images
print(f"Average time per image: {average_time_per_image_ms:.2f} ms")

Time taken for inference: 2386.35 ms
Number of images: 60
Average time per image: 39.77 ms


In [11]:
model = YOLO("models/yolov8n_openvino_model", task="detect")
results = model.val(data="coco128.yaml", task="detect", conf=0.25, iou=0.70, device="cpu", save_json=True, plots=True)

Ultralytics 8.3.118 🚀 Python-3.13.3 torch-2.7.0 CPU (Apple M2)
Loading models/yolov8n_openvino_model for OpenVINO inference...
Using OpenVINO LATENCY mode for batch=1 inference...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1713.3±1289.4 MB/s, size: 44.3 KB)


val: Scanning /Users/phuocle/Desktop/untitled folder/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]
/Users/phuocle/Desktop/Class Archive/Classes/CMPE 258/Homework/Homework 2/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:03<00:00,  2.51it/s]


                   all        128        929      0.666      0.492      0.614      0.485
                person         61        254      0.801      0.665      0.773       0.61
               bicycle          3          6      0.667      0.333      0.499       0.43
                   car         12         46      0.909      0.217      0.571      0.398
            motorcycle          4          5      0.667        0.8      0.825      0.704
              airplane          5          6        0.8      0.667        0.8      0.612
                   bus          5          7      0.556      0.714      0.771      0.746
                 train          3          3      0.667      0.667      0.777       0.65
                 truck          5         12          1       0.25      0.625      0.444
                  boat          2          6       0.25      0.167       0.27      0.134
         traffic light          4         14      0.667      0.143      0.429      0.386
             stop sig

## Inference Faster R-CNN

In [32]:
class fasterRCNN:
    """
    A class for detecting objects in images using pre-trained Faster R-CNN model.
    """
    
    # COCO dataset class names
    COCO_INSTANCE_CATEGORY_NAMES = [
        '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
        'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
        'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
        'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
        'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
        'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
        'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
        'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
        'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
        'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
        'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book',
        'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
    ]
    
    def __init__(self, confidence_threshold=0.7):
        """
        Initialize the detector with model and parameters.
        
        Args:
            confidence_threshold (float): Minimum confidence score for detections
        """
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")
        
        # Load pre-trained model
        self.model = fasterrcnn_resnet50_fpn(FasterRCNN_ResNet50_FPN_Weights.DEFAULT, box_score_thresh=confidence_threshold)
        self.model.to(self.device)
        self.model.eval()  # Set to evaluation mode
        
        # Set transform for image preprocessing
        self.transform = transforms.Compose([transforms.ToTensor()])
        
        # Set confidence threshold for filtering detections
        self.confidence_threshold = confidence_threshold
        
        # Generate color map for visualization
        self.color_map = self._generate_color_map()
    
    def _generate_color_map(self):
        """
        Generate a color map for visualization, assigning distinct colors to each class.
        
        Returns:
            dict: Mapping of class indices to colors
        """
        # Use some predefined colors for better distinction
        base_colors = list(mcolors.TABLEAU_COLORS.values())
        num_classes = len(self.COCO_INSTANCE_CATEGORY_NAMES)
        
        # If more colors are needed, add random colors
        if num_classes > len(base_colors):
            more_colors = [tuple(np.random.rand(3)) for _ in range(num_classes - len(base_colors))]
            color_list = base_colors + more_colors
        else:
            color_list = base_colors[:num_classes]
        
        return {i: color_list[i % len(color_list)] for i in range(num_classes)}
    
    def detect(self, image_path):
        """
        Detect objects in an image.
        
        Args:
            image_path (str): Path to the input image
            
        Returns:
            tuple: (image, boxes, scores, labels) where boxes, scores, and labels are filtered based on the confidence threshold
        """
        # Load and preprocess image
        image = Image.open(image_path)
        image_tensor = self.transform(image).to(self.device)
        
        # Run inference
        with torch.no_grad():
            prediction = self.model([image_tensor])
        
        # Process the prediction
        boxes = prediction[0]['boxes'].cpu().numpy()
        scores = prediction[0]['scores'].cpu().numpy()
        labels = prediction[0]['labels'].cpu().numpy()
        
        # Filter predictions based on confidence
        confident_detections = scores > self.confidence_threshold
        boxes = boxes[confident_detections]
        scores = scores[confident_detections]
        labels = labels[confident_detections]
        
        return image, boxes, scores, labels
    
    def visualize_detections(self, image, boxes, scores, labels, output_path=None):
        """
        Visualize detections on the image and optionally save the result.
        
        Args:
            image (PIL.Image): The input image
            boxes (np.ndarray): Bounding box coordinates [x1, y1, x2, y2]
            scores (np.ndarray): Confidence scores for each detection
            labels (np.ndarray): Class labels for each detection
            output_path (str, optional): Path to save the output image
            
        Returns:
            None: Displays the plot or saves it to output_path
        """
        # Create figure for plotting
        plt.figure(figsize=(12, 8))
        plt.imshow(np.array(image))
        
        # Draw each detection
        for box, score, label_idx in zip(boxes, scores, labels):
            x1, y1, x2, y2 = box
            
            # Get class name and color
            class_name = self.COCO_INSTANCE_CATEGORY_NAMES[label_idx]
            color = self.color_map[label_idx]
            
            # Draw bounding box
            plt.gca().add_patch(plt.Rectangle(
                (x1, y1), x2 - x1, y2 - y1, 
                fill=False, color=color, linewidth=2
            ))
            
            # Add text with class name and score
            plt.text(
                x1, y1 - 5, 
                f"{class_name}: {score:.2f}", 
                color='white', 
                bbox=dict(facecolor=color, alpha=0.7),
                fontsize=10
            )
        
        plt.title('Faster R-CNN Object Detection')
        plt.axis('off')
        plt.tight_layout()
        
        # Save or display the result
        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            plt.close()  # Close the figure to free memory
        else:
            plt.show()
    
    def process_image(self, image_path, output_folder="output/results/fasterrcnn"):
        """
        Process a single image: detect objects, visualize, and save results.
        
        Args:
            image_path (str): Path to the input image
            output_folder (str): Folder to save the output image
            
        Returns:
            str: Path to the saved output image
        """
        # Create output folder if it doesn't exist
        os.makedirs(output_folder, exist_ok=True)
        
        # Get filename without extension
        image_filename = os.path.basename(image_path)
        filename_no_ext = os.path.splitext(image_filename)[0]
        
        # Detect objects
        image, boxes, scores, labels = self.detect(image_path)
        
        # Generate output filename with timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = os.path.join(output_folder, f"{filename_no_ext}.png")
        
        # Visualize and save detections
        self.visualize_detections(image, boxes, scores, labels, output_path)
        
        # Return statistics
        detections = {
            'num_objects': len(boxes),
            'classes': [self.COCO_INSTANCE_CATEGORY_NAMES[label] for label in labels],
            'output_path': output_path
        }
        
        return detections
    
    def process_directory(self, input_dir, output_folder):
        """
        Process all images in a directory.
        
        Args:
            input_dir (str): Directory containing input images
            output_folder (str): Folder to save output images
            
        Returns:
            list: List of detection results for each image
        """
        results = []
        # Process each image in directory
        for filename in os.listdir(input_dir):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_path = os.path.join(input_dir, filename)
                result = self.process_image(image_path, output_folder)
                results.append({
                    'filename': filename,
                    'detections': result
                })
        
        return results

In [33]:
# Initialize the detector
detector = fasterRCNN(confidence_threshold=0.25)

# Process a directory of images
input_directory = "output/extracted"
output_directory = "output/results/fasterRCNN/fasterRCNN"
os.makedirs(output_directory, exist_ok=True)

# Start timer
start_time = time.time()

results = detector.process_directory(input_directory, output_directory)

# Print the time taken for inference
elapsed_time_ms = (time.time() - start_time) * 1000
print(f"Time taken for inference: {elapsed_time_ms:.2f} ms")

# Print number of images
print(f"Number of images: {len(results)}")

# Print average time per image
average_time_per_image_ms = elapsed_time_ms / len(results)
print(f"Average time per image: {average_time_per_image_ms:.2f} ms")

Using device: cpu


/Users/phuocle/Desktop/Class Archive/Classes/CMPE 258/Homework/Homework 2/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(


Time taken for inference: 87067.75 ms
Number of images: 60
Average time per image: 1451.13 ms


In [3]:
coco_annotation_path = "coco128/annotations/instances_train2017v2.json"
image_paths = "coco128/images/train2017"
temp_folder = "temp"

# Create temporary directory if it doesn't exist
os.makedirs(temp_folder, exist_ok=True)

# Load COCO annotations
coco_gt = COCO(coco_annotation_path)
image_ids = list(coco_gt.imgs.keys())

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)
model.eval()

preprocess = weights.transforms()

# Create a mapping from model category IDs to COCO category IDs
model_to_coco_category = {}
model_categories = weights.meta["categories"]

for coco_cat_id, cat_info in coco_gt.cats.items():
    coco_cat_name = cat_info['name']
    # Try to find a match in model categories (case insensitive)
    for model_cat_id, model_cat_name in enumerate(model_categories):
        if coco_cat_name.lower() == model_cat_name.lower():
            model_to_coco_category[model_cat_id] = coco_cat_id

coco_results = []

start_time = time.time()

for img_id in tqdm(image_ids):
    # Get image info
    img_info = coco_gt.imgs[img_id]
    file_name = img_info['file_name']
    image_path = os.path.join(image_paths, file_name)

    img = read_image(image_path)

    processed = preprocess(img)
    
    with torch.no_grad():
        prediction = model([processed])[0]
    
    boxes = prediction['boxes']
    scores = prediction['scores']
    labels = prediction['labels']
    
    # Format predictions for COCO evaluation using the category mapping
    for box, score, label in zip(boxes, scores, labels):
        model_label = label.item()
        
        # Skip if the model label doesn't map to a COCO category
        if model_label not in model_to_coco_category:
            continue
            
        # Get the corresponding COCO category ID
        coco_label = model_to_coco_category[model_label]
        
        # Convert box format from [x1, y1, x2, y2] to [x, y, width, height]
        x1, y1, x2, y2 = box.tolist()
        coco_box = [x1, y1, x2 - x1, y2 - y1]
        
        # Create COCO result with mapped category ID
        result = {
            'image_id': img_id,
            'category_id': coco_label,  # Use mapped COCO category ID
            'bbox': coco_box,
            'score': score.item()
        }
        coco_results.append(result)

# End timing
end_time = time.time()
total_time = end_time - start_time
average_time = total_time / len(image_ids)

print(f"\nProcessed {len(image_ids)} images.")
print(f"Total inference time: {total_time:.2f} seconds")
print(f"Average inference time per image: {average_time:.2f} seconds")
print(f"Generated {len(coco_results)} detections.")

# Save COCO results to file
results_file = os.path.join(temp_folder, "detection_results.json")
with open(results_file, 'w') as f:
    json.dump(coco_results, f)

# Evaluate using COCO API
print("\nPerforming COCO evaluation...")
try:
    coco_dt = coco_gt.loadRes(results_file)
    coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    # Extract mAP scores
    mAP50_95 = coco_eval.stats[0]  # AP @[ IoU=0.50:0.95 | area=all ]
    mAP50 = coco_eval.stats[1]     # AP @[ IoU=0.50 | area=all ]

    print(f"\nResults:")
    print(f"mAP @ IoU=0.50:0.95: {mAP50_95:.4f}")
    print(f"mAP @ IoU=0.50: {mAP50:.4f}")
except Exception as e:
    print(f"Error during evaluation: {e}")
    print("This might be due to category mismatches between model predictions and ground truth.")

# Clean up temporary files
if os.path.exists(results_file):
    os.remove(results_file)
    
# Try to remove temp directory if empty
try:
    os.rmdir(temp_folder)
except OSError:
    pass  # Directory not empty or other error, just leave it

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


100%|██████████| 128/128 [02:57<00:00,  1.39s/it]



Processed 128 images.
Total inference time: 177.53 seconds
Average inference time per image: 1.39 seconds
Generated 4307 detections.

Performing COCO evaluation...
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.37s).
Accumulating evaluation results...
DONE (t=0.18s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.494
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.739
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.530
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.303
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.564
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.638
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.378
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe

## Inference Faster R-CNN using Onnx Runtime

In [61]:
class ObjectDetector:
    COCO_INSTANCE_CATEGORY_NAMES = [
        '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
        'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
        'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
        'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
        'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
        'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
        'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
        'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
        'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
        'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
        'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book',
        'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
    ]
    
    def __init__(self, onnx_model_path="models/faster_rcnn.onnx", confidence_threshold=0.5):
        """
        Initialize the object detector with the ONNX model
        
        Args:
            onnx_model_path: Path to the exported ONNX model
            confidence_threshold: Threshold for filtering detections
        """
        self.onnx_model_path = onnx_model_path
        self.confidence_threshold = confidence_threshold
        self.label_map = {i: name for i, name in enumerate(COCO_INSTANCE_CATEGORY_NAMES)}
        
        # Load the ONNX model once
        print(f"Loading ONNX model from {onnx_model_path}...")
        start_time = time.time()
        self.session = ort.InferenceSession(onnx_model_path)
        self.input_name = self.session.get_inputs()[0].name
        self.output_names = [output.name for output in self.session.get_outputs()]
        print(f"Model loaded in {time.time() - start_time:.2f} seconds")
        
        # Prepare image transform
        self.transform = transforms.Compose([
            transforms.ToTensor(),  # Convert PIL Image to tensor
        ])
    
    def detect(self, image_path):
        """
        Run object detection on a single image
        
        Args:
            image_path: Path to the input image
            
        Returns:
            boxes, labels, scores: Detection results
        """
        # Load and preprocess the image
        image = Image.open(image_path).convert("RGB")
        input_tensor = self.transform(image)
        
        # Run inference with the ONNX model
        outputs = self.session.run(self.output_names, {self.input_name: input_tensor.numpy()})
        
        # Process the output
        boxes, labels, scores = outputs
        
        # Filter by confidence threshold
        keep = scores >= self.confidence_threshold
        boxes = boxes[keep]
        labels = labels[keep]
        scores = scores[keep]
        
        return boxes, labels, scores
    
    def visualize_and_save_detections(self, image_path, boxes, labels, scores, output_path):
        """
        Visualize detection results on the image and save to disk
        
        Args:
            image_path: Path to the input image
            boxes: Bounding boxes [N, 4] in format [x1, y1, x2, y2]
            labels: Class labels [N]
            scores: Confidence scores [N]
            output_path: Path to save the output image
        
        Returns:
            output_path: Path where the image was saved
        """
        # Load the image
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Draw the bounding boxes and labels
        for box, label, score in zip(boxes, labels, scores):
            x1, y1, x2, y2 = box.astype(int)
            class_name = self.label_map.get(label, f"Class {label}")
            
            # Draw bounding box
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            
            # Draw label and score
            text = f"{class_name}: {score:.2f}"
            cv2.putText(image, text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        # Convert back to BGR for saving with OpenCV
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # Save the image
        cv2.imwrite(output_path, image)
        # print(f"Saved detection results to {output_path}")
        
        return output_path
    
    def process_single_image(self, image_path, output_path):
        """
        Process a single image with detection and visualization
        
        Args:
            image_path: Path to input image
            output_path: Path to save output image
            
        Returns:
            detection_info: Dictionary with detection information
        """
        # Run detection
        boxes, labels, scores = self.detect(image_path)
        
        # Save visualization
        self.visualize_and_save_detections(image_path, boxes, labels, scores, output_path)
        
        # Print detection summary
        # print(f"Detected {len(boxes)} objects in {os.path.basename(image_path)}")
        # for i, (label, score) in enumerate(zip(labels, scores)):
        #     class_name = self.label_map.get(label, f"Class {label}")
        #     print(f"  {i+1}. {class_name}: {score:.2f}")
        
        return {
            "image": os.path.basename(image_path),
            # "detection_count": len(boxes)
        }

def process_image_folder(input_folder, output_folder, onnx_model_path="models/faster_rcnn.onnx", confidence_threshold=0.5):
    """
    Process all images in a folder using the object detection model
    
    Args:
        input_folder: Path to folder containing input images
        output_folder: Path to folder where output images will be saved
        onnx_model_path: Path to the ONNX model
        confidence_threshold: Threshold for filtering detections
    
    Returns:
        processed_files: List of processed image paths
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Get list of image files
    valid_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
    image_files = [f for f in os.listdir(input_folder) 
                if os.path.isfile(os.path.join(input_folder, f)) 
                and os.path.splitext(f)[1].lower() in valid_extensions]
    
    if not image_files:
        print(f"No valid image files found in {input_folder}")
        return []
    
    print(f"Found {len(image_files)} images to process")
    
    # Initialize the detector once (main optimization)
    detector = ObjectDetector(onnx_model_path, confidence_threshold)
    
    # Process all images sequentially
    start_time = time.time()
    processed_files = []
    # total_detections = 0
    
    for i, img_file in enumerate(image_files):
        # print(f"Processing image {i+1}/{len(image_files)}: {img_file}")
        input_path = os.path.join(input_folder, img_file)
        output_path = os.path.join(output_folder, f"detected_{img_file}")
        
        try:
            # Process the image
            result = detector.process_single_image(input_path, output_path)
            processed_files.append(output_path)
            # total_detections += result["detection_count"]
        except Exception as e:
            print(f"Error processing {img_file}: {str(e)}")
    
    # Print final summary
    total_time = time.time() - start_time
    print(f"\nProcessing complete: {len(processed_files)}/{len(image_files)} images processed successfully")
    print(f"Total processing time: {total_time:.2f} seconds (avg {total_time/len(image_files):.2f} sec per image)")
    
    # if processed_files:
    #     print(f"Total objects detected: {total_detections} (avg {total_detections/len(processed_files):.1f} per image)")
    
    return processed_files

if __name__ == "__main__":
    input_folder = "output/extracted"  # Folder containing images to process
    output_folder = "output/results/fasterRCNN/onnx"  # Folder where processed images will be saved
    onnx_model_path = "models/faster_rcnn.onnx"
    
    # Process the folder
    processed_files = process_image_folder(
        input_folder,
        output_folder,
        onnx_model_path=onnx_model_path,
        confidence_threshold=0.5
    )

Found 60 images to process
Loading ONNX model from models/faster_rcnn.onnx...
Model loaded in 0.28 seconds

Processing complete: 60/60 images processed successfully
Total processing time: 88.29 seconds (avg 1.47 sec per image)


In [9]:
coco_annotation_path = "coco128/annotations/instances_train2017v2.json"
image_paths = "coco128/images/train2017"
temp_folder = "temp"
onnx_model_path = "models/faster_rcnn.onnx"

# Create temp directory
os.makedirs(temp_folder, exist_ok=True)

# Load COCO annotations
coco_gt = COCO(coco_annotation_path)
image_ids = list(coco_gt.imgs.keys())

# Use weights for preprocessing and category mapping
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
preprocess = weights.transforms()
model_categories = weights.meta["categories"]

# Build category mapping
model_to_coco_category = {}
for coco_cat_id, cat_info in coco_gt.cats.items():
    coco_cat_name = cat_info['name']
    for model_cat_id, model_cat_name in enumerate(model_categories):
        if coco_cat_name.lower() == model_cat_name.lower():
            model_to_coco_category[model_cat_id] = coco_cat_id

# Load ONNX model
session = ort.InferenceSession(onnx_model_path, providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
output_names = ["boxes", "labels", "scores"]  # Defined during export

# Run inference and collect COCO results
coco_results = []
start_time = time.time()

for img_id in tqdm(image_ids):
    img_info = coco_gt.imgs[img_id]
    file_name = img_info['file_name']
    image_path = os.path.join(image_paths, file_name)

    img = read_image(image_path)
    if img.shape[0] == 1:  # Grayscale
        img = img.repeat(3, 1, 1) 
    processed = preprocess(img)
    input_tensor = processed.numpy()  # shape: (3, H, W)

    # Inference
    outputs = session.run(output_names, {input_name: input_tensor})
    boxes, labels, scores = outputs  # Each is a NumPy array

    for box, score, label in zip(boxes, scores, labels):
        model_label = int(label)
        if model_label not in model_to_coco_category:
            continue

        coco_label = model_to_coco_category[model_label]
        x1, y1, x2, y2 = box
        coco_box = [x1, y1, x2 - x1, y2 - y1]

        result = {
            "image_id": int(img_id),
            "category_id": int(coco_label),
            "bbox": [float(x) for x in coco_box],
            "score": float(score)
        }
        coco_results.append(result)

# Timing
end_time = time.time()
total_time = end_time - start_time
average_time = total_time / len(image_ids)

print(f"\nProcessed {len(image_ids)} images.")
print(f"Total inference time: {total_time:.2f} seconds")
print(f"Average inference time per image: {average_time:.2f} seconds")
print(f"Generated {len(coco_results)} detections.")

# Save results
results_file = os.path.join(temp_folder, "detection_results.json")
with open(results_file, 'w') as f:
    json.dump(coco_results, f)

# Evaluate
print("\nPerforming COCO evaluation...")
try:
    coco_dt = coco_gt.loadRes(results_file)
    coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    mAP50_95 = coco_eval.stats[0]
    mAP50 = coco_eval.stats[1]

    print(f"\nmAP @ IoU=0.50:0.95: {mAP50_95:.4f}")
    print(f"mAP @ IoU=0.50: {mAP50:.4f}")
except Exception as e:
    print(f"Error during evaluation: {e}")
    print("This might be due to mismatched categories or model output issues.")

# Clean-up
if os.path.exists(results_file):
    os.remove(results_file)
try:
    os.rmdir(temp_folder)
except OSError:
    pass

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


100%|██████████| 128/128 [02:51<00:00,  1.34s/it]



Processed 128 images.
Total inference time: 171.34 seconds
Average inference time per image: 1.34 seconds
Generated 4318 detections.

Performing COCO evaluation...
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.26s).
Accumulating evaluation results...
DONE (t=0.17s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.494
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.738
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.529
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.300
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.567
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.638
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.380
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe

## Inference Faster R-CNN using OpenVino Core

In [ ]:
class FasterRCNNDetector:
    """
    A class for performing object detection using a converted Faster R-CNN model in OpenVINO.
    Handles dynamic input shapes appropriately.
    """
    
    def __init__(self, model_path, device="CPU", confidence_threshold=0.5, nms_threshold=0.5):
        """
        Initialize the detector with a converted OpenVINO model.
        
        Args:
            model_path (str): Path to the OpenVINO IR model (.xml file)
            device (str): Device to run inference on ('CPU', 'GPU', etc.)
            confidence_threshold (float): Minimum confidence score for detections
            nms_threshold (float): NMS IoU threshold for filtering overlapping boxes
        """
        self.confidence_threshold = confidence_threshold
        self.nms_threshold = nms_threshold
        self.device = device
        self.scale_x = 1.0
        self.scale_y = 1.0
        
        # Load the model
        print(f"Loading model from {model_path}")
        self.core = ov.Core()
        
        # Check available devices
        available_devices = self.core.available_devices
        print(f"Available devices: {available_devices}")
        
        if device not in available_devices and device != "CPU":
            print(f"WARNING: {device} not available. Falling back to CPU.")
            device = "CPU"
        
        self.model = self.core.read_model(model_path)
        
        # Print model inputs and outputs for debugging
        print("Model inputs:")
        for input_layer in self.model.inputs:
            print(f"  - {input_layer.get_any_name()}: {input_layer.get_partial_shape()}")
        
        print("Model outputs:")
        for output_layer in self.model.outputs:
            print(f"  - {output_layer.get_any_name()}: {output_layer.get_partial_shape()}")
        
        # Get the input name
        self.input_name = self.model.inputs[0].get_any_name()
        
        # Initialize with fixed channel count (3) but dynamic height and width
        # This is critical for handling images of different sizes
        try:
            self.model.reshape({self.input_name: ov.PartialShape([3, -1, -1])})
            print("Model reshaped successfully with dynamic dimensions")
        except Exception as e:
            print(f"WARNING: Could not reshape model with dynamic dimensions: {e}")
            print("Using original model shape")
        
        # Configure performance settings
        config = {
            "PERFORMANCE_HINT": "LATENCY",  # Prioritize latency over throughput
            "CACHE_DIR": "./openvino_cache"  # Cache optimized model
        }
        
        # Compile the model
        print(f"Compiling model for {device}")
        try:
            self.compiled_model = self.core.compile_model(self.model, device, config)
            print("Model compiled successfully")
        except Exception as e:
            print(f"Error compiling model for {device}: {e}")
            print("Trying to compile without config")
            self.compiled_model = self.core.compile_model(self.model, device)
        
        # Get the output names for post-processing
        self.output_names = [output.get_any_name() for output in self.compiled_model.outputs]
        print(f"Model outputs: {self.output_names}")
        
        # COCO dataset classes for label mapping (80 classes)
        self.classes = [
            'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat',
            'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog',
            'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella',
            'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite',
            'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle',
            'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
            'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
            'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote',
            'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book',
            'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
        ]
        
    def preprocess_image(self, image, target_size=None):
        """
        Preprocess an image for inference.
        
        Args:
            image: Input image (numpy array in BGR format from OpenCV)
            target_size: Optional tuple (height, width) to resize image
            
        Returns:
            Preprocessed image ready for the model
        """
        # Convert from BGR to RGB
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Get original dimensions
        orig_height, orig_width = image_rgb.shape[:2]
        
        # Resize if target_size is specified
        if target_size:
            image_resized = cv2.resize(image_rgb, (target_size[1], target_size[0]))
            self.scale_x = orig_width / target_size[1]
            self.scale_y = orig_height / target_size[0]
        else:
            image_resized = image_rgb
            self.scale_x = 1.0
            self.scale_y = 1.0
        
        # Transpose from HWC to CHW format (height, width, channels) -> (channels, height, width)
        image_chw = image_resized.transpose(2, 0, 1)
        
        # Convert to float32 and normalize to [0,1]
        image_normalized = image_chw.astype(np.float32) / 255.0
        
        # Apply normalization using ImageNet mean and std
        # These values are used in torchvision's pretrained models
        mean = np.array([0.485, 0.456, 0.406]).reshape((3, 1, 1))
        std = np.array([0.229, 0.224, 0.225]).reshape((3, 1, 1))
        image_normalized = (image_normalized - mean) / std
        
        return image_normalized, (orig_height, orig_width)
        
    def detect(self, image, target_size=(800, 800)):
        """
        Perform object detection on an image.
        
        Args:
            image: Input image (numpy array in BGR format from OpenCV)
            target_size: Optional size to resize image to (height, width)
            
        Returns:
            List of detections with [xmin, ymin, xmax, ymax, score, class_id]
        """
        # Preprocess the image with target size
        preprocessed_image, (orig_height, orig_width) = self.preprocess_image(image, target_size)
        
        # Get input shape information
        input_shape = preprocessed_image.shape
        print(f"Input shape: {input_shape}")
        
        # Create a dictionary with the input tensor
        inputs = {self.input_name: preprocessed_image}
        
        # Perform inference
        print("Running inference...")
        start_time = time.time()
        results = self.compiled_model(inputs)
        inference_time = time.time() - start_time
        print(f"Inference time: {inference_time*1000:.2f} ms")
        
        # Debug output names and shapes
        print("Output keys/names:")
        for name in self.output_names:
            print(f"  - {name}: {results[name].shape}")
        
        # Extract outputs based on the model's output layer names
        # The output format depends on your exported model
        boxes = None
        labels = None
        scores = None
        
        # Try to identify outputs by name first
        for name in self.output_names:
            output_name_lower = name.lower()
            if "box" in output_name_lower:
                boxes = results[name]
            elif "label" in output_name_lower or "class" in output_name_lower:
                labels = results[name].astype(np.int64)
            elif "score" in output_name_lower or "conf" in output_name_lower:
                scores = results[name]
        
        # If name-based identification failed, try shape-based identification
        if boxes is None or labels is None or scores is None:
            for name in self.output_names:
                output = results[name]
                if output.ndim == 2 and output.shape[1] == 4:  # Boxes have shape [N, 4]
                    boxes = output
                elif output.ndim == 1:  # Could be scores or labels (1D)
                    if np.max(output) > 1.0:  # Likely labels (class IDs)
                        labels = output.astype(np.int64)
                    else:  # Likely confidence scores (0-1)
                        scores = output
        
        # Last resort: positional assignment
        if boxes is None or labels is None or scores is None:
            print("WARNING: Could not identify outputs by name or shape. Using position-based assignment.")
            # Try to handle any possible output format
            if len(self.output_names) >= 3:
                # Assume standard order: boxes, labels, scores
                boxes = results[self.output_names[0]]
                labels = results[self.output_names[1]].astype(np.int64)
                scores = results[self.output_names[2]]
            else:
                # Handle dictionary outputs (possible with some ONNX exports)
                for name in self.output_names:
                    output = results[name]
                    # Try to parse complex output structures
                    if isinstance(output, dict) and "boxes" in output and "labels" in output and "scores" in output:
                        boxes = output["boxes"]
                        labels = output["labels"].astype(np.int64)
                        scores = output["scores"]
                        break
        
        # Check if we have valid outputs
        if boxes is None or labels is None or scores is None:
            print("ERROR: Could not identify output format. Please check model export.")
            return []
        
        # Filter by confidence
        keep_indices = np.where(scores >= self.confidence_threshold)[0]
        
        # If no detections pass the threshold
        if len(keep_indices) == 0:
            return []
            
        filtered_boxes = boxes[keep_indices]
        filtered_labels = labels[keep_indices]
        filtered_scores = scores[keep_indices]
        
        # Scale boxes to original image dimensions
        # First check if boxes are normalized (between 0 and 1)
        is_normalized = np.all(filtered_boxes >= 0) and np.all(filtered_boxes <= 1)
        
        # Convert from model coordinates to original image coordinates
        detections = []
        for i in range(len(filtered_boxes)):
            xmin, ymin, xmax, ymax = filtered_boxes[i]
            
            # Handle normalized coordinates
            if is_normalized:
                # Convert from normalized [0,1] to pixel coordinates in target_size space
                xmin *= target_size[1]  # width
                ymin *= target_size[0]  # height
                xmax *= target_size[1]  # width
                ymax *= target_size[0]  # height
            
            # Scale back to original image dimensions
            xmin = int(xmin * self.scale_x)
            ymin = int(ymin * self.scale_y)
            xmax = int(xmax * self.scale_x)
            ymax = int(ymax * self.scale_y)
            
            # Ensure coordinates are within image bounds
            xmin = max(0, min(xmin, orig_width - 1))
            ymin = max(0, min(ymin, orig_height - 1))
            xmax = max(0, min(xmax, orig_width - 1))
            ymax = max(0, min(ymax, orig_height - 1))
            
            # Only add if we have a valid box
            if xmin < xmax and ymin < ymax:
                detections.append([
                    xmin, ymin, xmax, ymax,
                    float(filtered_scores[i]), int(filtered_labels[i])
                ])
            
        return detections
    
    def draw_detections(self, image, detections):
        """
        Draw detection boxes on the image with labels and scores.
        
        Args:
            image: Input image
            detections: List of detections [xmin, ymin, xmax, ymax, score, class_id]
            
        Returns:
            Image with drawn detections
        """
        result_image = image.copy()
        
        # Generate random colors for each class
        np.random.seed(42)  # For reproducibility
        colors = {i: tuple(map(int, np.random.randint(0, 255, 3))) for i in range(len(self.classes))}
        
        for det in detections:
            xmin, ymin, xmax, ymax, score, class_id = det
            
            # Get class name and color
            class_name = self.classes[class_id - 1] if 0 < class_id <= len(self.classes) else f"Unknown-{class_id}"
            color = colors.get(class_id, (0, 255, 0))  # Default to green if class_id not in colors
            
            # Draw bounding box
            cv2.rectangle(result_image, (xmin, ymin), (xmax, ymax), color, 2)
            
            # Prepare label text
            label = f"{class_name}: {score:.2f}"
            
            # Get text size
            (label_width, label_height), baseline = cv2.getTextSize(
                label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1
            )
            
            # Draw label background
            cv2.rectangle(
                result_image,
                (xmin, ymin - label_height - baseline - 5),
                (xmin + label_width, ymin),
                color,
                -1
            )
            
            # Draw label text
            cv2.putText(
                result_image,
                label,
                (xmin, ymin - baseline - 5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 0, 0),
                1
            )
            
        return result_image

def non_max_suppression(boxes, scores, iou_threshold=0.5):
    """
    Apply non-maximum suppression to avoid detecting too many
    overlapping bounding boxes for a given object.
    
    Args:
        boxes: Array of boxes coordinates [N, 4]
        scores: Array of confidence scores [N]
        iou_threshold: IoU threshold for NMS
        
    Returns:
        Indices of boxes to keep after NMS
    """
    # If no boxes, return empty list
    if len(boxes) == 0:
        return []
    
    # Convert to numpy arrays if not already
    boxes = np.array(boxes)
    scores = np.array(scores)
    
    # Boxes coordinates (xmin, ymin, xmax, ymax)
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]
    
    # Calculate areas of all boxes
    areas = (x2 - x1 + 1) * (y2 - y1 + 1)
    
    # Sort by confidence score
    order = scores.argsort()[::-1]
    
    keep = []
    while order.size > 0:
        # Pick the box with highest confidence score
        i = order[0]
        keep.append(i)
        
        # Calculate intersection with remaining boxes
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        
        # Calculate intersection area
        w = np.maximum(0.0, xx2 - xx1 + 1)
        h = np.maximum(0.0, yy2 - yy1 + 1)
        inter = w * h
        
        # Calculate IoU
        iou = inter / (areas[i] + areas[order[1:]] - inter)
        
        # Keep boxes with IoU less than threshold
        inds = np.where(iou <= iou_threshold)[0]
        order = order[inds + 1]
    
    return keep

def main():
    """
    Main function to demonstrate using the Faster R-CNN detector.
    """
    # Path to your OpenVINO IR model files (XML and BIN)
    model_path = "models/fasterrcnn_openvino_model/faster_rcnn_model.xml"
    
    # Path to test image
    image_path = "/Users/phuocle/Desktop/Class Archive/Classes/CMPE 258/Homework/Homework 2/output/extracted/frame_0-00-00.000_interval.jpg"    # Replace with your test image
    
    # Initialize the detector
    detector = FasterRCNNDetector(
        model_path=model_path,
        device="CPU",  # Change to "GPU" if available
        confidence_threshold=0.3,  # Lower threshold to detect more objects
        nms_threshold=0.7  # NMS threshold for filtering overlapping boxes
    )
    threshold = 0.3
    nms = 0.7
    size = "800,800"
    
    detector.confidence_threshold = threshold
    detector.nms_threshold = nms
    target_size = tuple(map(int, size.split(',')))
    
    # Check if image exists
    if not Path(image_path).exists():
        print(f"ERROR: Image not found at {image_path}")
        print("Please specify a valid image path or modify the code to use a webcam/video")
        return
    
    # Read the image
    image = cv2.imread(image_path)
    if image is None:
        print(f"ERROR: Could not read image from {image_path}")
        return
        
    # Perform detection with specified target size
    print(f"Running detection with target size {target_size}...")
    detections = detector.detect(image, target_size)
    print(f"Found {len(detections)} objects before NMS")
    
    # Apply non-max suppression to filter overlapping boxes
    if len(detections) > 0:
        boxes = np.array([det[:4] for det in detections])
        scores = np.array([det[4] for det in detections])
        keep_indices = non_max_suppression(boxes, scores, detector.nms_threshold)
        detections = [detections[i] for i in keep_indices]
    
    print(f"Found {len(detections)} objects after NMS")
    
    # Draw detections on the image
    result_image = detector.draw_detections(image, detections)
    
    # Save the output image
    output_path = "detection_result.jpg"
    cv2.imwrite(output_path, result_image)
    print(f"Results saved to {output_path}")

main()

Loading model from models/fasterrcnn_openvino_model/faster_rcnn_model.xml
Available devices: ['CPU']
Model inputs:
  - input: [3,?,?]
Model outputs:
  - boxes: [..100,4]
  - labels: [..100]
  - scores: [..100]
Model reshaped successfully with dynamic dimensions
Compiling model for CPU
Model compiled successfully
Model outputs: ['boxes', 'labels', 'scores']
Running detection with target size (640, 480)...
Input shape: (3, 640, 480)
Running inference...
Inference time: 805.93 ms
Output keys/names:
  - boxes: (10, 4)
  - labels: (10,)
  - scores: (10,)
Found 1 objects before NMS
Found 1 objects after NMS
Results saved to detection_result.jpg
